In [237]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [ ]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 3, 6
M = matroids.CompleteGraphic(n)
#M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())

frozenset({0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14})
deleting  0


In [239]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g, M):
    E = M.groundset()
    flats = list(M.lattice_of_flats())
    sum = P(0)
    for F in flats:
        sum += f(M.delete(E - F))(x) * g(M.contract(F))(x)
    return sum

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

def rightAugpoly(M):
    return convolve(chow, rev(kl_poly), M)

def leftAugpoly(M):
    return convolve(grev, chow_polynomial, M)

In [240]:
print("rank of M: ", r)
print("chow polynomial of M: ", chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
for F in flat_div['S']:
    print(F)

rank of M:  5
chow polynomial of M:  x^4 + 187*x^3 + 732*x^2 + 187*x + 1
x^4 - 14*x^3 + 71*x^2 - 154*x + 120
frozenset({10, 11, 14})
frozenset({11})
frozenset({11, 12})
frozenset({14})
frozenset({9, 14})
frozenset({9})
frozenset({9, 11, 13})
frozenset({10, 13})
frozenset({10})
frozenset({12, 13, 14})
frozenset({9, 10, 12})
frozenset({13})
frozenset()
frozenset({9, 10, 11, 12, 13, 14})
frozenset({12})


In [241]:


def deleconj(M, e, f):
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    sum2 = 0
    print("f(M):     ", og_matroid_poly)
    print("f(M\\e):  ", deleted_matroid_poly)
    for F in flat_div['S']:
        poly1 = chow_polynomial(M.contract(F.union({e})))
        poly2 = f(M.delete(M.groundset() - F))
        term = x * poly1 * poly2
        sum2 += term
        print(F)
        print("--------------------------------")
        print("poly1: ", poly1)
        print("poly2: ", poly2)
        print("term: ", term)
        print("--------------------------------")
        print("********************************")
    print("summation equals:", sum2)
    return P(deleted_matroid_poly + sum2)

def deleconjp2(M, e, f):
    sum2 = 0
    for F in flat_div['T']:
        poly1 = chow_polynomial(M.contract(F))
        poly2 = f(M.delete(M.groundset() - F -{e}))
        term = poly1 * poly2
        sum2 += term
    return P(sum2)

print("right", deleconj(M, e, rightAugpoly))

f(M):      1 + 218*x + 1643*x^2 + 1643*x^3 + 218*x^4 + x^5
f(M\e):   1 + 203*x + 1488*x^2 + 1488*x^3 + 203*x^4 + x^5
frozenset({10, 11, 14})
--------------------------------
poly1:  x + 1
poly2:  1 + 4*x + x^2
term:  x + 5*x^2 + 5*x^3 + x^4
--------------------------------
********************************
frozenset({11})
--------------------------------
poly1:  x^2 + 8*x + 1
poly2:  1 + x
term:  x + 9*x^2 + 9*x^3 + x^4
--------------------------------
********************************
frozenset({11, 12})
--------------------------------
poly1:  x + 1
poly2:  1 + 3*x + x^2
term:  x + 4*x^2 + 4*x^3 + x^4
--------------------------------
********************************
frozenset({14})
--------------------------------
poly1:  x^2 + 8*x + 1
poly2:  1 + x
term:  x + 9*x^2 + 9*x^3 + x^4
--------------------------------
********************************
frozenset({9, 14})
--------------------------------
poly1:  x + 1
poly2:  1 + 3*x + x^2
term:  x + 4*x^2 + 4*x^3 + x^4
------------------------

In [242]:
print(deleconjp2(M, e, rightAugpoly))

15 + 202*x + 428*x^2 + 202*x^3 + 15*x^4


In [243]:
print(rightAugpoly(M))
print(leftAugpoly(M))

1 + 218*x + 1643*x^2 + 1643*x^3 + 218*x^4 + x^5
1 + 202*x + 1412*x^2 + 1412*x^3 + 202*x^4 + x^5
